# Does the model still work after we shrink it for the chip?

This is the most important honest question about the whole project, so it
gets the first walkthrough.

To run a neural network on an FPGA cheaply, we don't use the 32-bit floating
point numbers a GPU uses. We use small **fixed-point** numbers — here, 18-bit
values with 12 bits after the binary point (we call it "Q6.12"). That's what
makes the hardware tiny and fast. But it's also lossy: every weight and every
intermediate value gets rounded. So the fair worry a reviewer will raise is:

> *"Your benchmarks show a 60-nanosecond GRU. Fine. But you trained it with
> random weights. Once you quantise a **real, trained** model down to
> fixed-point, does it still predict anything useful — or did you trade all
> the accuracy away for speed?"*

This notebook answers that with a real model on a real benchmark. The plan is
simple:

1. Take **Mackey–Glass**, the classic chaotic time-series used to test
   forecasters for decades.
2. Train an ordinary GRU (in PyTorch, normal floating point) to forecast it.
3. Run **that exact trained model** in the chip's Q6.12 fixed-point arithmetic
   and compare. If the accuracy survives, the compiler's deployment is safe.

We also run the fast diagonal-linear "SSM" model, so we can see the honest
speed-vs-accuracy trade across the architecture family.

In [1]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## 1. The benchmark: Mackey–Glass

Mackey–Glass is a delay differential equation. Its output looks like a smooth
wave that never quite repeats — it's *chaotic*, which is exactly why it's a
good forecasting test: you can't cheat by memorising, you have to model the
dynamics. It needs no download; we generate it from its own equation.

In [2]:
def mackey_glass(n, tau=17, beta=0.2, gamma=0.1, p=10):
    warmup = 1000
    x = np.full(n + warmup + tau, 1.2)
    for t in range(tau, len(x) - 1):
        x[t + 1] = x[t] + beta * x[t - tau] / (1 + x[t - tau] ** p) - gamma * x[t]
    return x[warmup:]

series = mackey_glass(12000)
series = (series - series.mean()) / series.std()   # zero-mean, unit-variance
print(f"generated {len(series)} points; first few: "
      f"{np.round(series[:4], 3)}")

generated 12017 points; first few: [0.31  0.187 0.088 0.012]


## 2. The task: forecast 15 steps into the future

We show the model the last **20** values and ask it to predict the value
**15 steps ahead**. Fifteen steps is far enough that the chaos has moved on —
the naive "tomorrow looks like today" guess falls apart — so the model's
memory genuinely has to do work. We hold out the last 30% as an unseen test
set.

In [3]:
L, HORIZON, H = 20, 15, 16

def windows(s):
    X = np.array([s[t - L:t] for t in range(L, len(s) - HORIZON)], np.float32)
    Y = np.array([s[t + HORIZON] for t in range(L, len(s) - HORIZON)], np.float32)
    return X, Y

X, Y = windows(series)
cut = int(0.7 * len(X))
Xtr, Ytr, Xte, Yte = X[:cut], Y[:cut], X[cut:], Y[cut:]

def nrmse(pred, target):
    """Normalised error: 0 is perfect, 1 means 'no better than the average'."""
    return float(np.sqrt(np.mean((pred - target) ** 2)) / target.std())

# The honest yardstick: just predict "the last value we saw".
persistence = nrmse(Xte[:, -1], Yte)
print(f"persistence baseline NRMSE = {persistence:.3f}  "
      f"(this is the 'do nothing clever' score to beat)")

persistence baseline NRMSE = 1.647  (this is the 'do nothing clever' score to beat)


## 3. Train a normal GRU (floating point)

Nothing special here — a small standard GRU with 16 hidden units, trained the
usual way. This is the "ground truth" model we'll later squeeze into
fixed-point.

In [4]:
class GRUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(1, H, batch_first=True)
        self.fc = nn.Linear(H, 1)

    def forward(self, x):
        out, _ = self.gru(x.unsqueeze(-1))
        return self.fc(out[:, -1, :]).squeeze(-1)

net = GRUNet()
opt = torch.optim.Adam(net.parameters(), lr=5e-3)
lossf = nn.MSELoss()
Xtr_t, Ytr_t = torch.tensor(Xtr), torch.tensor(Ytr)
for _ in range(200):
    opt.zero_grad()
    lossf(net(Xtr_t), Ytr_t).backward()
    opt.step()

net.eval()
with torch.no_grad():
    pred_float = net(torch.tensor(Xte)).numpy()
gru_float = nrmse(pred_float, Yte)
print(f"GRU (float) NRMSE = {gru_float:.4f}  "
      f"-> {persistence / gru_float:.0f}x better than persistence")

GRU (float) NRMSE = 0.0321  -> 51x better than persistence


## 4. The real test: run that model in the chip's fixed-point

Now we take the trained weights and re-run the GRU exactly the way the FPGA
would: every weight rounded onto the Q6.12 grid, every multiply-add truncated
the way the hardware truncates, and the `tanh`/`sigmoid` gates read from a
small lookup table (the chip has no floating-point `exp`). If the score barely
moves, the compiler's deployment preserves the model.

In [5]:
FRAC = 12
SCALE = float(1 << FRAC)
_HI = (1 << (5 + FRAC)) - 1

def q_weight(x):            # constants: round to the nearest grid point
    return np.clip(np.rint(np.asarray(x, np.float64) * SCALE), -_HI - 1, _HI) / SCALE

def q_trunc(x):            # arithmetic: truncate, exactly like the hardware
    return np.floor(np.asarray(x, np.float64) * SCALE) / SCALE

# a 512-entry tanh lookup table over [-4, 4] -- the chip's activation
_TAB = q_weight(np.tanh(-4 + (np.arange(512) + 0.5) * 8 / 512))
def lut_tanh(x):
    xc = np.clip(x, -4, 4 - 1e-9)
    return _TAB[np.minimum(((xc + 4) * 64).astype(int), 511)]
def lut_sigmoid(x):
    return q_trunc(0.5 + 0.5 * lut_tanh(q_trunc(0.5 * x)))

p = dict(net.named_parameters())
Wih = q_weight(p['gru.weight_ih_l0'].detach())
Whh = q_weight(p['gru.weight_hh_l0'].detach())
bih = q_weight(p['gru.bias_ih_l0'].detach())
bhh = q_weight(p['gru.bias_hh_l0'].detach())
Wfc, bfc = q_weight(p['fc.weight'].detach()), q_weight(p['fc.bias'].detach())

def gru_fixed_point(seq):
    h = np.zeros(H)
    for value in seq:
        x = q_weight(np.array([value]))
        gi, gh = q_trunc(Wih @ x + bih), q_trunc(Whh @ h + bhh)
        r = lut_sigmoid(q_trunc(gi[:H] + gh[:H]))
        z = lut_sigmoid(q_trunc(gi[H:2 * H] + gh[H:2 * H]))
        n = lut_tanh(q_trunc(gi[2 * H:] + q_trunc(r * gh[2 * H:])))
        h = q_trunc(q_trunc((1 - z) * n) + q_trunc(z * h))
    return q_trunc(Wfc @ h + bfc)[0]

pred_fixed = np.array([gru_fixed_point(x) for x in Xte])
gru_fixed = nrmse(pred_fixed, Yte)
print(f"GRU (Q6.12 fixed-point) NRMSE = {gru_fixed:.4f}")
print(f"accuracy kept: {gru_float / gru_fixed * 100:.0f}% of the float model")
print(f"largest single-prediction difference float vs fixed: "
      f"{np.max(np.abs(pred_float - pred_fixed)):.4f}")

GRU (Q6.12 fixed-point) NRMSE = 0.0323
accuracy kept: 99% of the float model
largest single-prediction difference float vs fixed: 0.0211


## 5. The fast lane: the diagonal-linear SSM

The compiler also has a much cheaper engine — the diagonal-linear state-space
model that runs at 20 ns/sample on just 87 DSPs (vs the GRU's 60 ns and 871).
It's a *linear* recurrence, so on a chaotic task we expect it to give some
accuracy back. Let's see how much — that's the honest trade a user gets to
make.

In [6]:
class DiagSSM(nn.Module):
    def __init__(self):
        super().__init__()
        self.logit_a = nn.Parameter(torch.zeros(H))
        self.B = nn.Parameter(torch.randn(H, 1) * 0.2)
        self.head = nn.Sequential(nn.Linear(H, H), nn.Tanh(), nn.Linear(H, 1))

    def forward(self, x):
        a = torch.sigmoid(self.logit_a)
        h = torch.zeros(x.shape[0], H)
        for t in range(x.shape[1]):
            h = a * h + x[:, t:t + 1] @ self.B.T
        return self.head(h).squeeze(-1)

ssm = DiagSSM()
opt2 = torch.optim.Adam(ssm.parameters(), lr=8e-3)
for _ in range(200):
    opt2.zero_grad()
    lossf(ssm(Xtr_t), Ytr_t).backward()
    opt2.step()
ssm.eval()
with torch.no_grad():
    ssm_float = nrmse(ssm(torch.tensor(Xte)).numpy(), Yte)
print(f"diagonal-linear SSM (float) NRMSE = {ssm_float:.4f}")

diagonal-linear SSM (float) NRMSE = 0.2092


## What this notebook demonstrates

| model | NRMSE (lower is better) | vs persistence | on hardware |
|---|---|---|---|
| persistence ("last value") | ~1.65 | — | — |
| **GRU, float** | **~0.032** | **~51× better** | 60 ns/sample |
| **GRU, Q6.12 fixed-point** | **~0.032** | **~51× better** | 60 ns/sample, 871 DSP |
| diagonal-linear SSM, float | ~0.21 | ~8× better | 20 ns/sample, 87 DSP |

Two things matter here, and both are good news:

1. **The fixed-point deployment is essentially free of accuracy loss.** The
   GRU squeezed into the chip's Q6.12 arithmetic scores the same as the full
   floating-point model (~99% of its quality). So the 60 ns/sample number is
   not a number for a toy — it's the speed of a model that genuinely forecasts
   a chaotic system ~51× better than the naive baseline, with its trained
   accuracy intact through the deployment.

2. **The architecture family gives a real dial.** If you need every drop of
   accuracy, the GRU is there at 60 ns. If you're chasing throughput or
   fitting many models on one board, the diagonal-linear engine is 3× faster
   and 10× smaller — but on a hard chaotic task it gives back accuracy
   (0.20 vs 0.03). That's an honest trade the compiler lets a user choose, not
   a hidden cost.

The one caveat, stated plainly: this is *simulation-accurate* fixed-point
(the same arithmetic the verified hardware executes). Confirming it on a
physical board — same weights, real silicon, measured error — is the next
phase of hardware validation.